## Part 1a: Why Naïve RAG Fails — A Live Demonstration

In [27]:
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-community langchain-huggingface numpy --quiet

In [28]:
from dotenv import load_dotenv
import os

load_dotenv()

# If .env is missing, manually enter your key
if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Setup complete.


In [29]:
# Our knowledge base — 5 documents
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",           # doc_0
    "BERT is a bidirectional encoder trained using masked language modelling.",               # doc_1
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",  # doc_2
    "Gradient descent is an optimization technique used to minimize the loss function.",      # doc_3
    "Neural networks learn by adjusting weights through backpropagation.",                    # doc_4
]

# Our query: deliberately uses a synonym that dense models handle OK, but BM25 struggles with
query_vocab_mismatch = "How do neural nets learn?"

# Our query: deliberately uses an exact keyword that BM25 excels at
query_exact_match = "BM25 term frequency"

print(f"Corpus size: {len(corpus)} documents")
print(f"Query 1 (semantic): '{query_vocab_mismatch}'")
print(f"Query 2 (keyword):  '{query_exact_match}'")

Corpus size: 5 documents
Query 1 (semantic): 'How do neural nets learn?'
Query 2 (keyword):  'BM25 term frequency'


In [30]:
# --- Naïve Dense-Only Retrieval ---
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

corpus_embeddings = model.encode(corpus, convert_to_numpy=True)

def dense_retrieve(query, top_k=3):
    q_vec = model.encode([query], convert_to_numpy=True)[0]
    scores = corpus_embeddings @ q_vec / (
        np.linalg.norm(corpus_embeddings, axis=1) * np.linalg.norm(q_vec)
    )
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i], corpus[i]) for i in ranked]

print("=== Dense Retrieval Results ===")
print(f"\nQuery: '{query_vocab_mismatch}'")
for rank, (idx, score, doc) in enumerate(dense_retrieve(query_vocab_mismatch), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

print(f"\nQuery: '{query_exact_match}'")
for rank, (idx, score, doc) in enumerate(dense_retrieve(query_exact_match), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== Dense Retrieval Results ===

Query: 'How do neural nets learn?'
  Rank 1 [doc_4] score=0.7550: Neural networks learn by adjusting weights through backpropagation.
  Rank 2 [doc_3] score=0.3656: Gradient descent is an optimization technique used to minimize the los
  Rank 3 [doc_1] score=0.2918: BERT is a bidirectional encoder trained using masked language modellin

Query: 'BM25 term frequency'
  Rank 1 [doc_2] score=0.5714: The BM25 algorithm ranks documents based on term frequency and inverse
  Rank 2 [doc_0] score=0.1448: Transformers use self-attention mechanisms to process sequences in par
  Rank 3 [doc_4] score=0.1129: Neural networks learn by adjusting weights through backpropagation.


## Part 1b: Sparse Retrieval — BM25

In [31]:
from rank_bm25 import BM25Okapi

# BM25 works on tokenized (word-split) documents
tokenized_corpus = [doc.lower().split() for doc in corpus]

bm25 = BM25Okapi(tokenized_corpus)

def bm25_retrieve(query, top_k=3):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    ranked = np.argsort(scores)[::-1][:top_k]
    return [(i, scores[i], corpus[i]) for i in ranked]

print("=== BM25 Retrieval Results ===")
print(f"\nQuery: '{query_vocab_mismatch}'")
for rank, (idx, score, doc) in enumerate(bm25_retrieve(query_vocab_mismatch), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

print(f"\nQuery: '{query_exact_match}'")
for rank, (idx, score, doc) in enumerate(bm25_retrieve(query_exact_match), 1):
    print(f"  Rank {rank} [doc_{idx}] score={score:.4f}: {doc[:70]}")

=== BM25 Retrieval Results ===

Query: 'How do neural nets learn?'
  Rank 1 [doc_4] score=1.2259: Neural networks learn by adjusting weights through backpropagation.
  Rank 2 [doc_3] score=0.0000: Gradient descent is an optimization technique used to minimize the los
  Rank 3 [doc_2] score=0.0000: The BM25 algorithm ranks documents based on term frequency and inverse

Query: 'BM25 term frequency'
  Rank 1 [doc_2] score=2.9625: The BM25 algorithm ranks documents based on term frequency and inverse
  Rank 2 [doc_4] score=0.0000: Neural networks learn by adjusting weights through backpropagation.
  Rank 3 [doc_3] score=0.0000: Gradient descent is an optimization technique used to minimize the los


## Part 1c: Dense Retrieval — SBERT and ColBERT

In [32]:
# ColBERT-style Late Interaction — implemented from scratch to understand the mechanism
# We use all-MiniLM as our encoder (true ColBERT uses a dedicated model, but the math is identical)

from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def colbert_score(query: str, document: str) -> float:
    """
    ColBERT MaxSim scoring.
    Each query token finds its best matching document token.
    Score = sum of those max similarities.
    """
    query_tokens = query.lower().split()       # tokenize (simplified: word-level)
    doc_tokens   = document.lower().split()

    # Encode each token as a vector
    q_vecs = encoder.encode(query_tokens, convert_to_numpy=True)   # shape: (|Q|, dim)
    d_vecs = encoder.encode(doc_tokens,   convert_to_numpy=True)   # shape: (|D|, dim)

    # Normalize
    q_vecs = q_vecs / np.linalg.norm(q_vecs, axis=1, keepdims=True)
    d_vecs = d_vecs / np.linalg.norm(d_vecs, axis=1, keepdims=True)

    # Similarity matrix: (|Q|, |D|)
    sim_matrix = q_vecs @ d_vecs.T

    # MaxSim: for each query token, take max across all doc tokens
    max_sims = sim_matrix.max(axis=1)    # shape: (|Q|,)

    return float(max_sims.sum())

print("=== ColBERT-style Late Interaction Scores ===")
query = "How do neural nets learn?"
print(f"Query: '{query}'\n")
scores = []
for i, doc in enumerate(corpus):
    s = colbert_score(query, doc)
    scores.append(s)
    print(f"  doc_{i} score={s:.4f}: {doc[:70]}")

best = np.argmax(scores)
print(f"\nTop document: doc_{best} — '{corpus[best]}'")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== ColBERT-style Late Interaction Scores ===
Query: 'How do neural nets learn?'

  doc_0 score=1.8954: Transformers use self-attention mechanisms to process sequences in par
  doc_1 score=2.1215: BERT is a bidirectional encoder trained using masked language modellin
  doc_2 score=1.8315: The BM25 algorithm ranks documents based on term frequency and inverse
  doc_3 score=1.9923: Gradient descent is an optimization technique used to minimize the los
  doc_4 score=3.3560: Neural networks learn by adjusting weights through backpropagation.

Top document: doc_4 — 'Neural networks learn by adjusting weights through backpropagation.'


## Part 1d: Hybrid Retrieval — BM25 + SBERT with Reciprocal Rank Fusion

In [33]:
# ============================================================
# NUMERICAL EXAMPLE: Full BM25 + SBERT + RRF walkthrough
# Query: "How do neural nets learn?"
# ============================================================

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "BERT is a bidirectional encoder trained using masked language modelling.",
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",
    "Gradient descent is an optimization technique used to minimize the loss function.",
    "Neural networks learn by adjusting weights through backpropagation.",
]

query = "How do neural nets learn?"

# ---- STEP 1: BM25 Scores ----
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(query.lower().split())

print("STEP 1: BM25 Raw Scores")
print("-" * 60)
for i, s in enumerate(bm25_scores):
    print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:60]}")

bm25_ranked = np.argsort(bm25_scores)[::-1]
bm25_ranks  = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked)}

print("\nBM25 Ranking (best → worst):", list(bm25_ranked))

STEP 1: BM25 Raw Scores
------------------------------------------------------------
  doc_0: 0.0000  |  Transformers use self-attention mechanisms to process sequen
  doc_1: 0.0000  |  BERT is a bidirectional encoder trained using masked languag
  doc_2: 0.0000  |  The BM25 algorithm ranks documents based on term frequency a
  doc_3: 0.0000  |  Gradient descent is an optimization technique used to minimi
  doc_4: 1.2259  |  Neural networks learn by adjusting weights through backpropa

BM25 Ranking (best → worst): [np.int64(4), np.int64(3), np.int64(2), np.int64(1), np.int64(0)]


In [34]:
# ---- STEP 2: SBERT Dense Scores ----
sbert = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

doc_vecs   = sbert.encode(corpus, convert_to_numpy=True)
query_vec  = sbert.encode([query], convert_to_numpy=True)[0]

doc_vecs_norm   = doc_vecs   / np.linalg.norm(doc_vecs,   axis=1, keepdims=True)
query_vec_norm  = query_vec  / np.linalg.norm(query_vec)

sbert_scores = doc_vecs_norm @ query_vec_norm

print("STEP 2: SBERT Cosine Scores")
print("-" * 60)
for i, s in enumerate(sbert_scores):
    print(f"  doc_{i}: {s:.4f}  |  {corpus[i][:60]}")

sbert_ranked = np.argsort(sbert_scores)[::-1]
sbert_ranks  = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked)}

print("\nSBERT Ranking (best → worst):", list(sbert_ranked))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


STEP 2: SBERT Cosine Scores
------------------------------------------------------------
  doc_0: 0.2637  |  Transformers use self-attention mechanisms to process sequen
  doc_1: 0.2918  |  BERT is a bidirectional encoder trained using masked languag
  doc_2: 0.0809  |  The BM25 algorithm ranks documents based on term frequency a
  doc_3: 0.3656  |  Gradient descent is an optimization technique used to minimi
  doc_4: 0.7550  |  Neural networks learn by adjusting weights through backpropa

SBERT Ranking (best → worst): [np.int64(4), np.int64(3), np.int64(1), np.int64(0), np.int64(2)]


In [35]:
# ---- STEP 3: Reciprocal Rank Fusion ----

K = 60  # RRF smoothing constant
print("STEP 3: Reciprocal Rank Fusion (k=60)")
print("-" * 80)
print(f"{'Doc':<8} {'BM25 rank':<12} {'SBERT rank':<13} {'RRF(BM25)':<14} {'RRF(SBERT)':<14} {'Total RRF'}")
print("-" * 80)

rrf_scores = {}
for doc_id in range(len(corpus)):
    rb = bm25_ranks[doc_id]
    rs = sbert_ranks[doc_id]
    rrf_bm25  = 1.0 / (K + rb)
    rrf_sbert = 1.0 / (K + rs)
    total     = rrf_bm25 + rrf_sbert
    rrf_scores[doc_id] = total
    print(f"  doc_{doc_id}  {rb:<12} {rs:<13} {rrf_bm25:<14.6f} {rrf_sbert:<14.6f} {total:.6f}")

# Final ranking
final_ranked = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

print("\n" + "=" * 80)
print("FINAL HYBRID RANKING")
print("=" * 80)
for rank, doc_id in enumerate(final_ranked, 1):
    print(f"  #{rank}  doc_{doc_id} (RRF={rrf_scores[doc_id]:.6f}): {corpus[doc_id]}")

STEP 3: Reciprocal Rank Fusion (k=60)
--------------------------------------------------------------------------------
Doc      BM25 rank    SBERT rank    RRF(BM25)      RRF(SBERT)     Total RRF
--------------------------------------------------------------------------------
  doc_0  5            4             0.015385       0.015625       0.031010
  doc_1  4            3             0.015625       0.015873       0.031498
  doc_2  3            5             0.015873       0.015385       0.031258
  doc_3  2            2             0.016129       0.016129       0.032258
  doc_4  1            1             0.016393       0.016393       0.032787

FINAL HYBRID RANKING
  #1  doc_4 (RRF=0.032787): Neural networks learn by adjusting weights through backpropagation.
  #2  doc_3 (RRF=0.032258): Gradient descent is an optimization technique used to minimize the loss function.
  #3  doc_1 (RRF=0.031498): BERT is a bidirectional encoder trained using masked language modelling.
  #4  doc_2 (RRF=0.0

In [36]:
# ---- Comparison Table: BM25 vs SBERT vs Hybrid ----

print("COMPARISON: BM25 vs SBERT vs Hybrid (for query: 'How do neural nets learn?')")
print("=" * 90)
print(f"{'Rank':<6} {'BM25 top doc':<50} {'SBERT top doc':<50}")
print("-" * 90)
for i in range(len(corpus)):
    bm25_doc = corpus[bm25_ranked[i]][:45]
    sbert_doc = corpus[sbert_ranked[i]][:45]
    print(f"  #{i+1}   {bm25_doc:<50} {sbert_doc:<50}")

print()
print("Hybrid Final Ranking:")
for rank, doc_id in enumerate(final_ranked, 1):
    print(f"  #{rank}  {corpus[doc_id]}")

COMPARISON: BM25 vs SBERT vs Hybrid (for query: 'How do neural nets learn?')
Rank   BM25 top doc                                       SBERT top doc                                     
------------------------------------------------------------------------------------------
  #1   Neural networks learn by adjusting weights th      Neural networks learn by adjusting weights th     
  #2   Gradient descent is an optimization technique      Gradient descent is an optimization technique     
  #3   The BM25 algorithm ranks documents based on t      BERT is a bidirectional encoder trained using     
  #4   BERT is a bidirectional encoder trained using      Transformers use self-attention mechanisms to     
  #5   Transformers use self-attention mechanisms to      The BM25 algorithm ranks documents based on t     

Hybrid Final Ranking:
  #1  Neural networks learn by adjusting weights through backpropagation.
  #2  Gradient descent is an optimization technique used to minimize the loss fun

### Second Numerical Example — Keyword-Heavy Query

In [37]:
# Test with a keyword-heavy query where BM25 should dominate

query2 = "BM25 term frequency ranking"

bm25_scores2  = bm25.get_scores(query2.lower().split())
bm25_ranked2  = np.argsort(bm25_scores2)[::-1]
bm25_ranks2   = {doc_id: rank+1 for rank, doc_id in enumerate(bm25_ranked2)}

query2_vec    = sbert.encode([query2], convert_to_numpy=True)[0]
query2_norm   = query2_vec / np.linalg.norm(query2_vec)
sbert_scores2 = doc_vecs_norm @ query2_norm
sbert_ranked2 = np.argsort(sbert_scores2)[::-1]
sbert_ranks2  = {doc_id: rank+1 for rank, doc_id in enumerate(sbert_ranked2)}

rrf_scores2 = {}
for doc_id in range(len(corpus)):
    rrf_scores2[doc_id] = 1/(K + bm25_ranks2[doc_id]) + 1/(K + sbert_ranks2[doc_id])
final_ranked2 = sorted(rrf_scores2, key=rrf_scores2.get, reverse=True)

print(f"Query: '{query2}'\n")
print(f"{'Doc':<8} {'BM25 rank':<12} {'SBERT rank':<13} {'RRF Score':<12} Document")
print("-" * 90)
for doc_id in range(len(corpus)):
    print(f"  doc_{doc_id}  {bm25_ranks2[doc_id]:<12} {sbert_ranks2[doc_id]:<13} {rrf_scores2[doc_id]:.6f}   {corpus[doc_id][:60]}")

print("\nHybrid Final Ranking:")
for rank, doc_id in enumerate(final_ranked2, 1):
    print(f"  #{rank}  doc_{doc_id}: {corpus[doc_id]}")

Query: 'BM25 term frequency ranking'

Doc      BM25 rank    SBERT rank    RRF Score    Document
------------------------------------------------------------------------------------------
  doc_0  5            5             0.030769   Transformers use self-attention mechanisms to process sequen
  doc_1  4            3             0.031498   BERT is a bidirectional encoder trained using masked languag
  doc_2  1            1             0.032787   The BM25 algorithm ranks documents based on term frequency a
  doc_3  3            4             0.031498   Gradient descent is an optimization technique used to minimi
  doc_4  2            2             0.032258   Neural networks learn by adjusting weights through backpropa

Hybrid Final Ranking:
  #1  doc_2: The BM25 algorithm ranks documents based on term frequency and inverse document frequency.
  #2  doc_4: Neural networks learn by adjusting weights through backpropagation.
  #3  doc_1: BERT is a bidirectional encoder trained using masked

## Part 1e: Building a Full Hybrid Retriever (Reusable Function)

In [38]:
class HybridRetriever:
    """
    A production-style Hybrid Retriever combining BM25 (sparse) + SBERT (dense)
    using Reciprocal Rank Fusion.
    """

    def __init__(self, corpus: list[str], sbert_model: str = "sentence-transformers/all-MiniLM-L6-v2", k: int = 60):
        self.corpus = corpus
        self.k = k

        # --- Build BM25 index ---
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25  = BM25Okapi(tokenized)

        # --- Build SBERT index ---
        self.sbert     = SentenceTransformer(sbert_model)
        doc_vecs       = self.sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs  = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)

    def retrieve(self, query: str, top_k: int = 3) -> list[dict]:
        # BM25
        bm25_scores  = self.bm25.get_scores(query.lower().split())
        bm25_ranked  = np.argsort(bm25_scores)[::-1]
        bm25_ranks   = {int(doc_id): rank+1 for rank, doc_id in enumerate(bm25_ranked)}

        # SBERT
        q_vec        = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec        = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(doc_id): rank+1 for rank, doc_id in enumerate(sbert_ranked)}

        # RRF Fusion
        rrf = {}
        for doc_id in range(len(self.corpus)):
            rrf[doc_id] = 1/(self.k + bm25_ranks[doc_id]) + 1/(self.k + sbert_ranks[doc_id])

        final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [
            {"doc_id": doc_id, "rrf_score": rrf[doc_id], "text": self.corpus[doc_id]}
            for doc_id in final
        ]

# Demo
retriever = HybridRetriever(corpus)

for q in ["How do neural nets learn?", "BM25 term frequency ranking", "attention in transformers"]:
    print(f"Query: '{q}'")
    results = retriever.retrieve(q, top_k=2)
    for r in results:
        print(f"  → doc_{r['doc_id']} (RRF={r['rrf_score']:.5f}): {r['text']}")
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Query: 'How do neural nets learn?'
  → doc_4 (RRF=0.03279): Neural networks learn by adjusting weights through backpropagation.
  → doc_3 (RRF=0.03226): Gradient descent is an optimization technique used to minimize the loss function.

Query: 'BM25 term frequency ranking'
  → doc_2 (RRF=0.03279): The BM25 algorithm ranks documents based on term frequency and inverse document frequency.
  → doc_4 (RRF=0.03226): Neural networks learn by adjusting weights through backpropagation.

Query: 'attention in transformers'
  → doc_0 (RRF=0.03279): Transformers use self-attention mechanisms to process sequences in parallel.
  → doc_4 (RRF=0.03226): Neural networks learn by adjusting weights through backpropagation.

